# KOSPI 기초통계량

`kospi_clean.csv`를 불러와 2010-01-01부터 2025-12-31까지의 수익률, 외국인 순매수비율, 기관 순매수비율, 개인 순매수비율 기초통계량을 확인합니다.

In [10]:
from pathlib import Path

import pandas as pd

def find_kospi_csv() -> Path:
    candidate_paths = [
        Path("..") / "시계열 자료" / "kospi_clean.csv",
        Path.cwd().parent / "시계열 자료" / "kospi_clean.csv",
        Path("시계열 자료") / "kospi_clean.csv",
        Path.cwd() / "시계열 자료" / "kospi_clean.csv",
    ]
    for path in candidate_paths:
        resolved = path.resolve()
        if resolved.exists():
            return resolved

    matches = sorted(Path.cwd().rglob("kospi_clean.csv"))
    if not matches:
        raise FileNotFoundError("kospi_clean.csv 파일을 찾지 못했습니다.")
    return matches[0].resolve()


data_path = find_kospi_csv()

kospi = pd.read_csv(data_path, parse_dates=["date"])
kospi = kospi[(kospi["date"] >= "2010-01-01") & (kospi["date"] <= "2025-12-31")].copy()
print(f"Loaded: {data_path}")
print(f"Period: {kospi['date'].min().date()} ~ {kospi['date'].max().date()} / rows={len(kospi):,}")
kospi.head()

Loaded: 시계열 자료\kospi_clean.csv
Period: 2010-01-04 ~ 2025-12-30 / rows=3,939


,market,date,open_index,high_index,low_index,close_index,return_pct,volume,turnover,krx_mkt_cap,...,n_unchanged,foreign_net_buy_ratio,inst_net_buy_ratio,personal_net_buy_ratio,securities_net_buy_ratio,insurance_net_buy_ratio,trust_net_buy_ratio,bank_net_buy_ratio,other_fin_net_buy_ratio,pension_net_buy_ratio
254,KOSPI,2010-01-04,1681.71,1696.14,1681.71,1696.14,0.79,295432853,4358188027372,894123953000000,...,95,0.054762,0.000067,-0.061947,-0.001224,0.012406,-0.003567,0.001777,-0.000139,-0.000779
255,KOSPI,2010-01-05,1701.62,1702.39,1686.45,1690.62,-0.33,407472890,6821415717586,891277501000000,...,113,0.057407,-0.030433,-0.024416,-0.002129,0.000219,-0.011967,0.001540,0.000036,-0.008560
256,KOSPI,2010-01-06,1697.88,1706.89,1696.10,1705.32,0.87,425158650,6386366250289,899021709000000,...,108,0.054936,-0.007642,-0.054336,0.005400,-0.003884,-0.009151,0.001267,-0.000873,-0.000899
257,KOSPI,2010-01-07,1702.92,1707.90,1683.45,1683.45,-1.28,461257505,7492783225429,887503216000000,...,107,0.029384,-0.012668,-0.022100,0.004168,-0.008130,-0.005932,0.001433,0.001604,-0.005110
258,KOSPI,2010-01-08,1694.06,1695.26,1668.84,1695.26,0.70,379028815,6960198086930,893475022000000,...,136,0.008541,-0.010402,0.005391,0.000439,0.001209,-0.011332,-0.005389,0.001764,0.004249


In [11]:
stats_cols = {
    "return_pct": "수익률",
    "foreign_net_buy_ratio": "외국인 순매수비율",
    "inst_net_buy_ratio": "기관 순매수비율",
    "personal_net_buy_ratio": "개인 순매수비율",
}

from scipy.stats import ttest_1samp

selected = kospi[list(stats_cols)]
t_stats = selected.apply(lambda col: ttest_1samp(col.dropna(), popmean=0.0).statistic)
p_values = selected.apply(lambda col: ttest_1samp(col.dropna(), popmean=0.0).pvalue)

basic_stats = pd.DataFrame({
    "mean": selected.mean(),
    "t-value": t_stats,
    "p-value": p_values,
    "std": selected.std(),
    "min": selected.min(),
    "max": selected.max(),
    "kurtosis": selected.kurt(),
    "skewness": selected.skew(),
})
basic_stats.index = [stats_cols[col] for col in basic_stats.index]
basic_stats

,mean,t-value,p-value,std,min,max,kurtosis,skewness
수익률,0.029185,1.700203,0.089172,1.077341,-8.770000,8.600000,6.349426,-0.277345
외국인 순매수비율,0.001013,1.458314,0.144834,0.043578,-0.199525,0.240183,1.575621,-0.066587
기관 순매수비율,-0.000202,-0.323060,0.746667,0.039273,-0.221802,0.263644,1.619346,0.203079
개인 순매수비율,-0.001650,-2.123041,0.033813,0.048771,-0.219275,0.192411,1.162066,-0.180403


In [12]:
from IPython.display import display

named_selected = selected.rename(columns=stats_cols)
cov_matrix = named_selected.cov()
corr_matrix = named_selected.corr()

print("Covariance Matrix")
display(cov_matrix)
print("Correlation Matrix")
display(corr_matrix)

Covariance Matrix


,수익률,외국인 순매수비율,기관 순매수비율,개인 순매수비율
수익률,1.160663,0.021873,0.012105,-0.034423
외국인 순매수비율,0.021873,0.001899,-0.000484,-0.001305
기관 순매수비율,0.012105,-0.000484,0.001542,-0.001021
개인 순매수비율,-0.034423,-0.001305,-0.001021,0.002379


Correlation Matrix


,수익률,외국인 순매수비율,기관 순매수비율,개인 순매수비율
수익률,1.000000,0.465893,0.286114,-0.655143
외국인 순매수비율,0.465893,1.000000,-0.282615,-0.614003
기관 순매수비율,0.286114,-0.282615,1.000000,-0.532874
개인 순매수비율,-0.655143,-0.614003,-0.532874,1.000000


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager, rcParams

preferred_fonts = ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Noto Sans KR"]
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
selected_font = None
for font_name in preferred_fonts:
    if font_name in available_fonts:
        selected_font = font_name
        break

sns.set_theme(style="whitegrid")
if selected_font is not None:
    rcParams["font.family"] = "sans-serif"
    rcParams["font.sans-serif"] = [selected_font]
rcParams["axes.unicode_minus"] = False
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

return_series = selected["return_pct"].dropna()
sns.histplot(return_series, bins=40, kde=True, ax=axes[0], color="#4C72B0", edgecolor="white")
axes[0].axvline(0, color="#D62728", linestyle="--", linewidth=1.2, label="0")
axes[0].axvline(return_series.mean(), color="#2CA02C", linestyle=":", linewidth=1.5, label="평균")
axes[0].set_title("수익률 분포", fontsize=13)
axes[0].set_xlabel("수익률")
axes[0].set_ylabel("빈도")
axes[0].legend()

flow_cols = ["foreign_net_buy_ratio", "inst_net_buy_ratio", "personal_net_buy_ratio"]
flow_colors = {
    "foreign_net_buy_ratio": "#1f77b4",
    "inst_net_buy_ratio": "#ff7f0e",
    "personal_net_buy_ratio": "#2ca02c",
}

for col in flow_cols:
    series = selected[col].dropna()
    sns.kdeplot(series, ax=axes[1], label=stats_cols[col], color=flow_colors[col], linewidth=2)
    axes[1].axvline(series.mean(), color=flow_colors[col], linestyle=":", linewidth=1.2)

axes[1].axvline(0, color="#D62728", linestyle="--", linewidth=1.2, label="0")
axes[1].set_title("순매수비율 비교 분포", fontsize=13)
axes[1].set_xlabel("순매수비율")
axes[1].set_ylabel("밀도")
axes[1].legend(title="변수")

fig.suptitle("KOSPI 수익률 및 수급비율 분포", fontsize=16, y=1.02)
fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()
